# PDF Question and Answer — RAG walkthrough

This notebook follows the Flask app's current document pipeline: extract PDF pages, retain page metadata, split text into chunks, retrieve relevant chunks with FAISS, and answer with TinyBERT.

The answer helper prints the model's score and the cited PDF page. A score is a confidence signal, not proof that an answer is correct. Compare scores for questions answered by the PDF and questions it does not answer before choosing any fallback threshold.

The app uses the Transformers `question-answering` pipeline, so `requirement.txt` pins `transformers==4.57.6`; Transformers 5 removed this pipeline.


In [ ]:
from pathlib import Path
import logging

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, pipeline

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("rag-notebook")


In [ ]:
# This example PDF is included in the project's PDF_files folder.
# Change this path if you want to try a different PDF.
PDF_PATH = Path("PDF_files") / "Artificial_intelligence.pdf"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH.resolve()}")


In [ ]:
# Extract text one page at a time and retain the original page number.
reader = PdfReader(str(PDF_PATH))
pages = []

for page_number, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    if text.strip():
        pages.append(
            Document(
                page_content=text,
                metadata={
                    "source": PDF_PATH.name,
                    "page": page_number,  # PDF page indexes start at 0.
                },
            )
        )

if not pages:
    raise ValueError("No readable text was extracted from this PDF.")

print(f"Extracted readable text from {len(pages)} pages.")


In [ ]:
# Keep 50 characters from the previous chunk to preserve nearby context.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
docs = text_splitter.split_documents(pages)
print(f"Created {len(docs)} text chunks.")


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-l6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False},
)


In [ ]:
# FAISS stores each chunk's vector and its original page metadata.
vector_store = FAISS.from_documents(docs, embeddings)


In [ ]:
ANSWER_MODEL_NAME = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(ANSWER_MODEL_NAME)
question_answerer = pipeline(
    "question-answering",
    model=ANSWER_MODEL_NAME,
    tokenizer=tokenizer,
)


In [ ]:
def answer_question(question: str) -> dict:
    """Answer from the best retrieved chunk and report its page and score."""
    matches = vector_store.similarity_search(question, k=3)
    if not matches:
        return {
            "question": question,
            "answer": "No relevant text was found in the PDF.",
            "score": None,
            "source_page": None,
        }

    best_chunk = matches[0]
    result = question_answerer(
        question=question,
        context=best_chunk.page_content,
    )
    score = float(result.get("score", 0.0))
    source_page = int(best_chunk.metadata.get("page", 0)) + 1

    logger.info("Answer score: %.3f | source page: %s", score, source_page)
    return {
        "question": question,
        "answer": result["answer"],
        "score": score,
        "source_page": source_page,
    }


In [ ]:
# Try an answerable question, then inspect the returned score and source page.
answer_question("What is intelligence?")


## Compare scores before adding a fallback

Add three questions whose answers are clearly in the PDF and three questions whose answers are not in it. Run `answer_question(question)` for each. Record whether the answer is correct along with its score. Scores can overlap, so don't choose a cutoff until you have reviewed examples.


In [ ]:
# Replace these examples with questions tailored to this PDF.
evaluation_questions = [
    # "A question whose answer is clearly in the PDF",
    # "A question whose answer is not in the PDF",
]

for test_question in evaluation_questions:
    answer_question(test_question)
